In [1]:
import datetime
import pandas as pd
from pandas import DataFrame
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from itertools import tee, product
import warnings
import re
from functools import reduce
import json
from typing import Literal
from datetime import datetime, date
warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)

def iterator(L):
	a, b = tee(L)
	next(b, None)
	return list(zip(a,b))


In [2]:
import os
r = re.compile(r'.*tph.*')
tphfiles = list(filter(r.match, os.listdir('october_data/agg_data')))
rushfiles = tphfiles[10:]
dayfiles = tphfiles[:10]

In [17]:
bad_stops = {}
for file in tphfiles:
	tph = pd.read_csv(f'october_data/agg_data/{file}')
	what = tph[(tph != 0).all(1)].index.values
	off = np.add(what,1)
	for old, new in zip(what,off):
		tph.iloc[old] = tph.iloc[new]
	if len(tph[(tph==0).all(1) | (tph != 0).all(1)]) == 0:
		tph.to_csv(f'october_data/agg_fixed/{file}')
	else: print(file)

In [18]:
headers = list(zip(tphfiles, ['day'] *10 + ['rush']*10))

In [19]:
for file,time in headers:
	tph_avg = pd.read_csv(f'october_data/agg_fixed/{file}',index_col=0).drop('dir',axis=1,errors='ignore')
	base_tph = pd.read_csv(f'october_data/{time}_tph_base.csv',index_col=0)
	agd = {k: lambda x: np.nanmean(x, axis=0, dtype=np.float64) for k in tph_avg.columns.values[1:]}
	w3 = tph_avg.groupby('stop').agg(agd)
	for i in w3[(w3==0).all(1)].index.values:
		try:
			w3.loc[i] = base_tph.loc[i]
		except KeyError:
			continue
	if not np.array_equal(w3[(w3==0).all(1)].index.values, ['140','H19','N12']):
		print(w3[(w3==0).all(1)].index.values)
		print(file, 'failed sanity check')
		break
	else:
		w3.drop(['140','H19','N12'],inplace=True)
		w3.to_csv(f'october_data/agg_fixed/{file}')

In [30]:
r = re.compile(r'.*delay.*')
delayfiles = list(filter(r.match, os.listdir('october_data/agg_data')))

for file in delayfiles:
	delay_file = pd.read_csv(f'october_data/agg_data/{file}',index_col='stop').mins
	delay_file = delay_file[delay_file.notna()]
	delay_file.to_csv(f'october_data/agg_fixed/{file}')

In [3]:
r = re.compile(r'.*tph.*')
tphfiles = list(filter(r.match, os.listdir('october_data/agg_data')))

In [22]:
for file in tphfiles[4:5]:
	delay_file = pd.read_csv(f'october_data/agg_data/{file}',index_col='stop').drop('dir',axis=1)

In [23]:
delay_file

,1,2,3,4,5,6,6X,7,7X,A,...,GS,H,J,L,M,N,Q,R,W,Z
stop,,,,,,,,,,,,,,,,,,,,,
101,11.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
101,11.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
103,11.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
103,11.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
104,11.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S01,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
S03,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
S03,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
(delay_file == 0).all()

1     False
2     False
3     False
4     False
5     False
6     False
6X    False
7     False
7X    False
A     False
B     False
C     False
D     False
E     False
F     False
FS    False
FX    False
G     False
GS    False
H     False
J     False
L     False
M     False
N     False
Q     False
R     False
W     False
Z     False
dtype: bool